# Import Packages

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import sys
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
import folium
import json
import time
import numpy as np
import h3
from folium.plugins import HeatMap
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
import time

In [2]:
sys.path.append('src')

In [3]:
from fetch_data import *
from data_io import *
from data_prep import *
from scoring import *
from threshold_clustering import *
from dbscan_clustering import *
from visualization import *

# User Inputs

In [4]:
# need to update it to circular boundaries based on user defined center and radius

#define boundaries
ATLANTA_BBOX  = [33.64, -84.55, 33.89, -84.29]

#need to make it scalable for more features later on

user_weights = {
    'restaurant': 0.5,
    'park': 0.1,
    'clinic': 0.4
}

#define radius of the search area in km, optional - can also define center
user_radius_km = 12

# Query Data

In [5]:
all_pois = []
all_pois = query_restaurant_data(ATLANTA_BBOX, all_pois)
all_pois = query_park_data(ATLANTA_BBOX, all_pois)
all_pois = query_hospital_and_clinic_data(ATLANTA_BBOX, all_pois)
print(f"Total POIs fetched: {len(all_pois)}")

Fetching restaurant data
 Found 1034 restaurant
Fetching cafe data
 Found 187 cafe
Fetching park data
 Found 505 park
Fetching hospital data
 Found 18 hospital
Fetching clinic data
 Found 50 clinic
Total POIs fetched: 1794


# Query Save & Load

In [6]:
name_of_the_file = "atlanta_pois"

In [7]:
save_pois(all_pois, name_of_the_file)

Saved as GeoJSON
Saved as CSV


In [8]:
df_pois = load_pois(name_of_the_file)

Loaded 1794
Summary by type:
type
restaurant    1034
park           505
cafe           187
clinic          50
hospital        18
Name: count, dtype: int64


# Data Prep - Data Points

In [9]:
# either use boundaries or radius function to create hex grids
# hexagons = create_hex_grids_with_boundaries(df_pois)
hexagons = create_hex_grids_with_radius(df_pois, radius_km=user_radius_km, size_of_grid = 8)



Using circular boundary: center (33.7490, -84.3880), radius 12 km
Generated 890 hexagons (before filtering)
Filtered to 536 hexagons within 12 km of center


# Data Prep - Features

In [10]:
df_hexagons = calculate_accessibility_scores(hexagons, df_pois)

Calculating accessibility scores for each hexagon...
  Processing hexagon 0/536...
  Processing hexagon 50/536...
  Processing hexagon 100/536...
  Processing hexagon 150/536...
  Processing hexagon 200/536...
  Processing hexagon 250/536...
  Processing hexagon 300/536...
  Processing hexagon 350/536...
  Processing hexagon 400/536...
  Processing hexagon 450/536...
  Processing hexagon 500/536...

✓ Calculated accessibility scores for 536 hexagons

Accessibility Score Statistics:
       restaurant_accessibility  park_accessibility  clinic_accessibility
count                536.000000          536.000000            536.000000
mean                   5.687573            5.254388              1.622551
std                    9.441404            3.973329              1.665690
min                    0.000000            0.154927              0.000000
25%                    0.124571            2.358554              0.434191
50%                    1.723031            4.127623              0.91

In [11]:
# df_hexagons = apply_user_weights(df_hexagons, user_weights)

df_hexagons = apply_user_weights(df_hexagons, user_weights, smooth_before_weighting=True, neighbor_weight=0.3)



print(df_hexagons.head())


APPLYING USER WEIGHTS
User preferences: {'restaurant': 0.5, 'park': 0.1, 'clinic': 0.4}
Sum of weights: 1.00 (should be 1.0)

Applying spatial smoothing to 3 score columns...
Neighbor weight: 0.30
  Smoothing hexagon 0/536...
  Smoothing hexagon 50/536...
  Smoothing hexagon 100/536...
  Smoothing hexagon 150/536...
  Smoothing hexagon 200/536...
  Smoothing hexagon 250/536...
  Smoothing hexagon 300/536...
  Smoothing hexagon 350/536...
  Smoothing hexagon 400/536...
  Smoothing hexagon 450/536...
  Smoothing hexagon 500/536...
✓ Spatial smoothing complete

User Match Score Statistics:
count    536.000000
mean       0.160917
std        0.176197
min        0.000952
25%        0.045058
50%        0.084837
75%        0.239125
max        0.951950
Name: user_match_score, dtype: float64
            hex_id        lat        lon  restaurant_accessibility  \
0  8844c1a165fffff  33.673920 -84.443727                  3.040909   
1  8844c1ad51fffff  33.833910 -84.453864                  1.100732

# Experiment 1: Threshold Clustering

In [12]:
df_threshold = cluster_based_on_score(df_hexagons)
df_threshold.head()


Thresholds: High = 0.182, Medium = 0.057

Suitability Distribution:
suitability_label
Okay             182
Less Suitable    177
Most Suitable    177
Name: count, dtype: int64

SUITABILITY TIER CHARACTERISTICS

Most Suitable (177 hexagons):
  Match Score Range: 0.182 - 0.952
  Avg restaurant Access: 14.393
  Avg park Access: 9.451
  Avg clinic Access: 3.578

Okay (182 hexagons):
  Match Score Range: 0.057 - 0.182
  Avg restaurant Access: 2.479
  Avg park Access: 4.101
  Avg clinic Access: 1.036

Less Suitable (177 hexagons):
  Match Score Range: 0.001 - 0.057
  Avg restaurant Access: 0.306
  Avg park Access: 2.263
  Avg clinic Access: 0.273


,hex_id,lat,lon,restaurant_accessibility,park_accessibility,clinic_accessibility,restaurant_norm,park_norm,clinic_norm,user_match_score,suitability,suitability_label
0,8844c1a165fffff,33.673920,-84.443727,3.040909,5.393015,0.666186,0.051335,0.263360,0.089415,0.087769,1,Okay
1,8844c1ad51fffff,33.833910,-84.453864,1.100732,0.980604,0.514872,0.018582,0.038340,0.069106,0.040767,2,Less Suitable
2,8844c1ab09fffff,33.672781,-84.344778,0.000000,1.094371,0.000000,0.000000,0.044142,0.000000,0.004414,2,Less Suitable
3,8844c106dbfffff,33.770312,-84.301629,17.036869,9.506511,2.963406,0.287606,0.473137,0.397747,0.350215,0,Most Suitable
4,8844c1ab21fffff,33.684779,-84.315839,0.893548,1.924718,0.000000,0.015084,0.086487,0.000000,0.016191,2,Less Suitable


In [13]:
threshold_map_name = "data/output_data/atlanta_threshold_map.html"
map_threshold = create_suitability_map(df_threshold, user_weights)
map_threshold.save(threshold_map_name)

Adding hexagons to map...
  Added 0/536 hexagons...
  Added 50/536 hexagons...
  Added 100/536 hexagons...
  Added 150/536 hexagons...
  Added 200/536 hexagons...
  Added 250/536 hexagons...
  Added 300/536 hexagons...
  Added 350/536 hexagons...
  Added 400/536 hexagons...
  Added 450/536 hexagons...
  Added 500/536 hexagons...


# Experiment 2: DBSCAN Clustering

In [14]:
df_dbscan = dbscan_score_clustering(df_hexagons, eps=0.1, min_samples=3)


DBSCAN CLUSTERING (Score-Based)
Parameters: eps=0.1, min_samples=3
Score range after scaling: [0.000, 1.000]

Results:
  Clusters found: 1
  Noise points: 0
  Cluster 0: 536 hexagons, avg score = 0.161


In [15]:
cluster_colors = get_cluster_colors(df_dbscan)

In [16]:
dbscan_map_name = "data/output_data/atlanta_dbscan_map.html"
map_dbscan = create_dbscan_map(df_dbscan, user_weights, cluster_colors=cluster_colors, use_heatmap=True, heatmap_radius=15)
map_dbscan.save(dbscan_map_name)

Adding hexagons to map...
  Added 0/536 hexagons...
  Added 50/536 hexagons...
  Added 100/536 hexagons...
  Added 150/536 hexagons...
  Added 200/536 hexagons...
  Added 250/536 hexagons...
  Added 300/536 hexagons...
  Added 350/536 hexagons...
  Added 400/536 hexagons...
  Added 450/536 hexagons...
  Added 500/536 hexagons...
Adding heatmap overlay for smooth visualization...


# Experiment 3 - Sptial DBSCAN

In [17]:
df_dbscan_spatial = dbscan_spatial_clustering(
    df_hexagons,
    eps=0.2,
    min_samples=3,
    spatial_weight=0.4
)


DBSCAN CLUSTERING (Spatially-Aware)
Parameters: eps=0.2, min_samples=3, spatial_weight=0.4

Results:
  Clusters found: 6
  Noise points: 1
  Noise/Uncertain: 1 hexagons
  Region 0: 502 hexagons, avg score = 0.127, extent = 33.1 km
  Region 1: 5 hexagons, avg score = 0.783, extent = 3.7 km
  Region 2: 15 hexagons, avg score = 0.590, extent = 7.3 km
  Region 3: 5 hexagons, avg score = 0.699, extent = 4.9 km
  Region 4: 5 hexagons, avg score = 0.899, extent = 2.6 km
  Region 5: 3 hexagons, avg score = 0.521, extent = 3.4 km


In [18]:
cluster_colors_spatial = get_cluster_colors(df_dbscan_spatial)

In [19]:
map_dbscan_spatial = create_dbscan_map(
    df_dbscan_spatial,
    user_weights,
    cluster_colors=cluster_colors_spatial,
    use_heatmap=True,
    heatmap_radius=15
)

spatial_dbscan_map_name = "data/output_data/atlanta_spatial_dbscan_map.html"
map_dbscan_spatial.save(spatial_dbscan_map_name)

Adding hexagons to map...
  Added 0/536 hexagons...
  Added 50/536 hexagons...
  Added 100/536 hexagons...
  Added 150/536 hexagons...
  Added 200/536 hexagons...
  Added 250/536 hexagons...
  Added 300/536 hexagons...
  Added 350/536 hexagons...
  Added 400/536 hexagons...
  Added 450/536 hexagons...
  Added 500/536 hexagons...
Adding heatmap overlay for smooth visualization...
